# Bathymetric Data

This notebook merges [GEBCO](https://www.gebco.net/) global bathymetric grids onto AIS tracks using `aisdb.webdata.bathymetry.Gebco`, so that each position report in a track carries the depth of water beneath the vessel at that moment. It is the runnable companion of the [Bathymetric Data](https://aisviz.gitbook.io/documentation/tutorials/bathymetric-data) GitBook page, which walks through the same steps in more narrative detail.

Knowing the depth under a vessel adds context that AIS alone cannot give. A ship holding close to the 20-meter contour is doing something very different from one crossing the open Atlantic, and that distinction only becomes visible once bathymetry is joined onto the track.

**What you will learn**

- How `Gebco` manages its raster cache in a `data_dir` and downloads GEBCO tiles on first use
- How to merge depth values onto AIS tracks with `Gebco.merge_tracks`
- How to bucket tracks into depth classes and color them for `aisdb.web_interface.visualize`


In [ ]:
# %pip install aisdb
# Running on Google Colab? Uncomment the line above to install AISdb first.


In [ ]:
import os
from datetime import datetime

import aisdb
from aisdb import SQLiteDBConn, DBQuery, DomainFromPoints
from aisdb.webdata.bathymetry import Gebco


## Staging the GEBCO bathymetry grid

`Gebco` wraps GEBCO's raster tiles behind a single interface. Give it a `data_dir` and it checks whether the tiles are already on disk, downloading and extracting them if not. The helper asserts that `data_dir` exists, so create it first with `os.makedirs`.

Be honest with yourself about the first run. GEBCO's global bathymetry ships as a single 7-Zip archive on AISdb's data server, and that archive is over 2.3 GB. The first time you point `Gebco` at an empty `data_dir`, expect a multi-minute download over a decent connection, followed by extraction with the system `7z` binary if it is installed, or `py7zr` as a pure-Python fallback otherwise. After that first run the tiles stay on disk, so every later run against the same `data_dir` skips the download and opens in a couple of seconds. Using a persistent cache directory such as `./raster_data/` (rather than a temp folder) means that speedup carries across sessions, not just within one script run.


In [ ]:
# set the path to the raster cache directory
data_dir = "./raster_data/"
os.makedirs(data_dir, exist_ok=True)

# opening Gebco as a context manager triggers the download only if the
# raster tiles are not already present in data_dir; once cached, this
# is fast on every subsequent run
with Gebco(data_dir=data_dir) as bathy:
    print("Bathymetry rasters ready:", list(bathy.rasterfiles.keys()))


## Querying a short AIS window

With the bathymetry grid staged, the next step is a normal AISdb query against the bundled `example_data.db`. That database covers the Gulf of Maine for January 2022, so a two-day slice is enough to get several tracks without pulling in the whole month.


In [ ]:
dbpath = "./example_data.db"
start_time = datetime.strptime("2022-01-01 00:00:00", "%Y-%m-%d %H:%M:%S")
end_time = datetime.strptime("2022-01-03 00:00:00", "%Y-%m-%d %H:%M:%S")

# Gulf of Maine bounding box covered by example_data.db
domain = DomainFromPoints(points=[(-68.5, 42.8)], radial_distances=[250000])

with SQLiteDBConn(dbpath=dbpath) as dbconn:
    qry = DBQuery(
        dbconn=dbconn,
        start=start_time,
        end=end_time,
        xmin=domain.boundary["xmin"], xmax=domain.boundary["xmax"],
        ymin=domain.boundary["ymin"], ymax=domain.boundary["ymax"],
        callback=aisdb.database.sqlfcn_callbacks.in_time_bbox_validmmsi,
    )
    tracks = list(aisdb.track_gen.TrackGen(qry.gen_qry(), decimate=False))

print("tracks retrieved:", len(tracks))


## Merging depth onto the tracks

`Gebco.merge_tracks` takes the track list and, for every position, looks up which raster tile covers that coordinate, loading it on demand, then reads the depth value at that point. It appends a `depth_metres` column to each track and adds it to the track's `dynamic` field set, so it travels along with `lon`, `lat`, and `time`.

A single track can cross more than one raster tile, so `Gebco` keeps every tile it has touched open for the lifetime of the `with` block and closes them all on exit. That is why the merge has to happen inside the same context manager, or right after it, rather than being deferred to later in the script.

GEBCO reports depth as a positive number below sea level, so a track's `depth_metres` array climbs from near zero at the shelf edge into the thousands over open ocean. A position that lands on a land pixel in the raster, a harbor entrance hugging the coast for instance, can come back negative, since that value is elevation above sea level rather than depth.


In [ ]:
with Gebco(data_dir=data_dir) as bathy:
    tracks_depth = list(bathy.merge_tracks(tracks))

# inspect the depth profile of the first track as a sanity check
sample_track = tracks_depth[0]
print("mmsi:", sample_track["mmsi"])
print("depth_metres sample:", sample_track["depth_metres"][:10])


## Coloring the tracks by depth

Depth values from `Gebco` are returned in meters below sea level. We use that to bucket each track into a rough depth class and set a `color` field, which `aisdb.web_interface.visualize` reads directly. These bands loosely follow real ocean depth zones: the continental shelf around the Gulf of Maine rarely exceeds 200 meters, the slope drops off quickly beyond it, and abyssal depths in the North Atlantic run a few thousand meters deep before the rare trench pushes past 6,000.


In [ ]:
def add_color(tracks):
    for track in tracks:
        # average depth across all positions in the track
        avg_depth = sum(track["depth_metres"]) / len(track["depth_metres"])

        if avg_depth <= 200:
            track["color"] = "yellow"   # continental shelf
        elif avg_depth <= 2000:
            track["color"] = "orange"   # continental slope
        elif avg_depth <= 6000:
            track["color"] = "pink"     # abyssal plain
        else:
            track["color"] = "red"      # deep ocean trench

        yield track


tracks_colored = list(add_color(tracks_depth))
print("colored tracks:", len(tracks_colored))


## Visualizing the colored tracks

`aisdb.web_interface.visualize` starts a local web server and opens a browser tab to render the tracks, which is not something a headless test run should trigger. Flip `SHOW_MAP` to `True` when running this notebook locally to see the depth-colored map; leave it `False` for unattended runs.


In [ ]:
SHOW_MAP = False  # set to True to open the interactive map in a browser

if SHOW_MAP:
    import nest_asyncio
    nest_asyncio.apply()

    aisdb.web_interface.visualize(
        tracks_colored,
        domain=domain,
        visualearth=True,
        open_browser=True,
    )
else:
    print("SHOW_MAP is False, skipping the interactive map. Flip it to True to view the tracks.")


## Next steps

This closes out the AISViz tutorial series. Head back to [7. Using Your AIS Data](7-using-your-ais-data.ipynb) or [8. Export Data to CSV](8-export-data-to-csv.ipynb) if you want to revisit exporting and downstream analysis, or continue exploring the [Bathymetric Data](https://aisviz.gitbook.io/documentation/tutorials/bathymetric-data) GitBook page for the full narrative walkthrough, including PortDist, ShoreDist, and CoastDist raster helpers that follow the same `data_dir` caching pattern shown here.
